In [0]:
from pyspark.sql import functions as F

REF = '/Volumes/voebem/bronze/arquivos/referencias'

SEM_ASPAS = chr(0)

In [0]:
aerodromos = (
    spark.read.format('csv')
    .option('sep', ';')
    .option('header', True)
    .option('skipRows', 1)
    .option('encoding', 'ISO-8859-1')
    .option('quote', SEM_ASPAS)
    .load(f'{REF}/AerodromosPublicos.csv')
)

aerodromos = aerodromos.select(
    F.col("`Código OACI`").alias('icao'),
    F.col('CIAD').alias('ciad'),
    F.col('Nome').alias('nome'),
    F.col('`Município`').alias('municipio'),
    F.col('UF').alias('uf'),
    F.col('`Município Servido`').alias('municipio_servido'),
    F.col('`UF Servido`').alias('uf_servido'),
    F.col('Latitude').alias('latitude'),
    F.col('Longitude').alias('longitude'),
    F.col('Altitude').alias('altitude'),
    F.col('`Situação`').alias('situacao')
).withColumn('_ingerido_em', F.current_timestamp())

aerodromos.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('voebem.bronze.aerodromos')

qtd = spark.table('voebem.bronze.aerodromos').count()

print(f"bronze.aerodromos: {qtd:,} linhas")

display(
    spark.sql("""
        SELECT icao, nome, municipio, uf
        FROM voebem.bronze.aerodromos
        WHERE icao IN ('SBRB', 'SBGR', 'SBSP', 'SBFZ')
    """)
)

In [0]:
def ler_empresas(arquivo: str):
    """
    Lê um cadastro de empresas. Sem união, sem enriquecimento:
    uma tabela por arquivo.
    """

    return (
        spark.read.format('csv')
        .option('sep', ';')
        .option('header', True)
        .option('skipRows', 1)
        .option('encoding', 'UTF-8')
        .option('quote', '"')
        .load(f'{REF}/{arquivo}')
        .select(
            F.col('ICAO').alias('icao'),
            F.col('Estrangeira').alias('sigla_iata'),
            F.col('Razao').alias('razao_social'),
            F.col('Servico').alias('servico'),
            F.col('Cidade').alias('cidade'),
            F.col('UF').alias('uf'),
            F.col('Ativa').alias('situacao'),
        )
    )


for arquivo, tabela in [
    ('pda_empresas_aereas_nacionais.csv', 'voebem.bronze.empresas_nacionais'),
    ('pda_empresas_aereas_estrangeiros.csv', 'voebem.bronze.empresas_estrangeiras'),
]:
    
    ler_empresas(arquivo).write \
        .format('delta') \
        .mode('overwrite') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(tabela)

    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

In [0]:
display(spark.sql("""
    SELECT
        'empresas_nacionais' AS tabela,
        COUNT(*) AS linhas,
        COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS icao
    FROM voebem.bronze.empresas_nacionais

    UNION ALL

    SELECT
        'empresas_estrangeiras' AS tabela,
        COUNT(*) AS linhas,
        COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS icao
    FROM voebem.bronze.empresas_estrangeiras
"""))

In [0]:
display(spark.sql(
    """
    SELECT icao, razao_social, servico, uf, situacao
    FROM voebem.bronze.empresas_nacionais
    WHERE icao IN ('GLO', 'TAM', 'AZU', 'PAM')
    ORDER BY icao
    """
))

In [0]:
display(spark.sql(
    """
    SELECT icao, razao_social, servico, uf, situacao
    FROM voebem.bronze.empresas_estrangeiras
    WHERE icao IN ('AAL', 'TAP', 'AVA', 'ARG')
    ORDER BY icao
    """
))